In [0]:
source_path='/Volumes/finguard/source/fraud_watchlist/source_data/'

In [0]:
dbutils.fs.ls(source_path)

[FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260813_120516_260801_0.json', name='fraud_watchlist_20260813_120516_260801_0.json', size=365, modificationTime=1786622717000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260813_120522_007447_1.json', name='fraud_watchlist_20260813_120522_007447_1.json', size=379, modificationTime=1786622723000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260813_120527_564767_2.json', name='fraud_watchlist_20260813_120527_564767_2.json', size=380, modificationTime=1786622728000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260813_120533_030644_3.json', name='fraud_watchlist_20260813_120533_030644_3.json', size=383, modificationTime=1786622734000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260813_120538_673202_4.json', na

In [0]:
input_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/finguard/source/fraud_watchlist/schema/")
    .option("cloudFiles.inferColumnTypes","true")
    .load(source_path)
)

In [0]:
from pyspark.sql import functions as F
transform_df = input_stream.select(
    "*",
    F.col("_metadata.file_path").alias("file_path"),
    F.current_timestamp().alias("ingestion_timestamp")
)

In [0]:
streaming_query = (transform_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation", "/Volumes/finguard/source/fraud_watchlist/checkpoint/")
.trigger(availableNow= True)
.toTable("finguard.bronze.fraud_watchlist_batch_test")
)
